In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Estimator の入力と出力

<Accordion>
<AccordionItem title="パッケージ・バージョン">

このページのコードは、以下の要件を使用して開発されました。
これらのバージョン以降の使用をお勧めします。

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>

このページでは、IBM Quantum&reg; コンピュート・リソース上でワークロードを実行する Qiskit Runtime Estimator プリミティブの入力と出力の概要を説明します。Estimator では、[**プリミティブ統合ブロック（PUB）**](/guides/primitive-input-output#pubs) と呼ばれるデータ構造を使用してベクトル化されたワークロードを効率的に定義できます。これらは Estimator プリミティブの [`run()`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2#run) メソッドへの入力として使用され、定義されたワークロードをジョブとして実行します。その後、ジョブが完了すると、使用した PUB とプリミティブから指定されたランタイム・オプションの両方に依存する形式で結果が返されます。

## 入力
各 PUB の形式は次のとおりです。

(`<単一回路>`, `<1 つまたは複数のオブザーバブル>`, `<オプションの 1 つまたは複数のパラメーター値>`, `<オプションの精度>`),

オプションの `parameter values` はリストまたは単一のパラメーターです。オブザーバブルとパラメーター値の要素は、[プリミティブの入力と出力](primitive-input-output#broadcasting-rules)のトピックで説明されている NumPy ブロードキャスティング・ルールに従って組み合わされ、ブロードキャストされた形状の各要素に対して 1 つの期待値推定が返されます。

> **Note:** 入力に測定値が含まれている場合、それらは無視されます。

Estimator プリミティブの PUB には最大で 4 つの値を含めることができます。
- 1 つ以上の [`Parameter`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.Parameter) オブジェクトを含む可能性のある単一の `QuantumCircuit`
- 推定する期待値を指定する 1 つ以上のオブザーバブルのリスト（配列に配置されます。例：0 次元配列として表現される単一のオブザーバブル、1 次元配列としてのオブザーバブルのリスト、など）。データは `ObservablesArrayLike` 形式（`Pauli`、`SparsePauliOp`、`PauliList`、または `str`）のいずれかで指定できます。
   > **Note:** - **同じ PUB 内**の可換オブザーバブルは、[このメソッド](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.PauliList#group_qubit_wise_commuting)を使用してグループ化されます。
>    - 異なる PUB 内の可換オブザーバブルは、同じ回路を持っていても、同じ測定を使用して推定されません。各 PUB は異なる測定基底を表し、したがって各 PUB に対して個別の測定が必要です。
>    - 可換オブザーバブルが同じ測定を使用して推定されるようにするには、同じ PUB 内にグループ化してください。
- 回路に対してバインドするパラメーター値のコレクション。最後のインデックスが回路の `Parameter` オブジェクトに対応する単一の配列のようなオブジェクトとして指定するか、回路に `Parameter` オブジェクトがない場合は省略する（または等価に `None` に設定する）ことができます。
- （オプション）推定する期待値の目標精度
---

以下のコードは、`Estimator` プリミティブへのベクトル化された入力のセットの例を示し、単一の `RuntimeJobV2` オブジェクトとして IBM&reg; バックエンドで実行します。

In [1]:
from qiskit.circuit import (
    Parameter,
    QuantumCircuit,
)
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp

from qiskit_ibm_runtime import (
    QiskitRuntimeService,
    EstimatorV2 as Estimator,
)

import numpy as np

# Instantiate runtime service and get
# the least busy backend
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Define a circuit with two parameters.
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.ry(Parameter("a"), 0)
circuit.rz(Parameter("b"), 0)
circuit.cx(0, 1)
circuit.h(0)

# Transpile the circuit
pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
transpiled_circuit = pm.run(circuit)
layout = transpiled_circuit.layout

# Now define a sweep over parameter values, the last axis of dimension 2 is
# for the two parameters "a" and "b"
params = np.vstack(
    [
        np.linspace(-np.pi, np.pi, 100),
        np.linspace(-4 * np.pi, 4 * np.pi, 100),
    ]
).T

# Define three observables. The inner length-1 lists cause this array of
# observables to have shape (3, 1), rather than shape (3,) if they were
# omitted.
observables = [
    [SparsePauliOp(["XX", "IY"], [0.5, 0.5])],
    [SparsePauliOp("XX")],
    [SparsePauliOp("IY")],
]
# Apply the same layout as the transpiled circuit.
observables = [
    [observable.apply_layout(layout) for observable in observable_set]
    for observable_set in observables
]

# Estimate the expectation value for all 300 combinations of observables
# and parameter values, where the pub result will have shape (3, 100).
#
# This shape is due to our array of parameter bindings having shape
# (100, 2), combined with our array of observables having shape (3, 1).
estimator_pub = (transpiled_circuit, observables, params)

# Instantiate the new Estimator object, then run the transpiled circuit
# using the set of parameters and observables.
estimator = Estimator(mode=backend)
job = estimator.run([estimator_pub])
result = job.result()

## 出力
1 つ以上の PUB が実行のために QPU に送信され、ジョブが正常に完了すると、データは `RuntimeJobV2.result()` メソッドを呼び出すことでアクセスできる [`PrimitiveResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult) コンテナー・オブジェクトとして返されます。

`PrimitiveResult` には、各 PUB の実行結果を含む [`PubResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult) オブジェクトのイテラブル・リストが含まれています。

このリストの各要素は、プリミティブの `run()` メソッドに送信された各 PUB に対応しています（例えば、20 個の PUB で送信されたジョブは、各 PUB に対応する 20 個の `PubResult` オブジェクトのリストを含む `PrimitiveResult` オブジェクトを返します）。

Estimator プリミティブの各 `PubResult` には、少なくとも期待値の配列（`PubResult.data.evs`）と関連する標準偏差（使用する `resilience_level` に応じて `PubResult.data.stds` または `PubResult.data.ensemble_standard_error`）が含まれていますが、指定されたエラー緩和オプションによってはより多くのデータが含まれる場合があります。

各 `PubResult` オブジェクトは `data` と `metadata` の両方の属性を持ちます。
    - `data` 属性は実際の測定値、標準偏差などを含むカスタマイズされた [`DataBin`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.DataBin) です。
    - `DataBin` は、関連する PUB の形状または構造、およびジョブを送信するために使用されたプリミティブで指定されたエラー緩和オプション（例：[ZNE](/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne) または [PEC](/guides/error-mitigation-and-suppression-techniques#probabilistic-error-cancellation-pec)）に応じてさまざまな属性を持ちます。
    - `metadata` 属性には、使用されたランタイムとエラー緩和オプションに関する情報が含まれています（このページの後半の[結果メタデータ](#result-metadata)セクションで説明）。

以下は Estimator 出力の `PrimitiveResult` データ構造の視覚的な概要です。

    ```
    └── PrimitiveResult
        ├── PubResult[0]
        │   ├── metadata
        │   └── data  ## In the form of a DataBin object
        │       ├── evs
        │       │   └── List of estimated expectation values in the shape
        |       |         specified by the first pub
        │       └── stds
        │           └── List of calculated standard deviations in the
        |                 same shape as above
        ├── PubResult[1]
        |   ├── metadata
        |   └── data  ## In the form of a DataBin object
        |       ├── evs
        |       │   └── List of estimated expectation values in the shape
        |       |        specified by the second pub
        |       └── stds
        |           └── List of calculated standard deviations in the
        |                same shape as above
        ├── ...
        ├── ...
        └── ...
    ```

簡単に言うと、単一のジョブは `PrimitiveResult` オブジェクトを返し、1 つ以上の `PubResult` オブジェクトのリストを含みます。これらの `PubResult` オブジェクトは、ジョブに送信された各 PUB の測定データを格納します。

以下のコード・スニペットは、上記で作成したジョブの `PrimitiveResult`（および関連する `PubResult`）の形式を説明しています。

In [2]:
print(
    f"The result of the submitted job had {len(result)} PUB and has a value:\n {result}\n"
)
print(
    f"The associated PubResult of this job has the following data bins:\n {result[0].data}\n"
)
print(f"And this DataBin has attributes: {result[0].data.keys()}")
print(
    "Recall that this shape is due to our array of parameter binding sets having shape (100, 2) -- where 2 is the\n\
         number of parameters in the circuit -- combined with our array of observables having shape (3, 1). \n"
)
with np.printoptions(threshold=200):
    print(
        f"The expectation values measured from this PUB are: \n{result[0].data.evs}"
    )

The result of the submitted job had 1 PUB and has a value:
 PrimitiveResult([PubResult(data=DataBin(evs=np.ndarray(<shape=(3, 100), dtype=float64>), stds=np.ndarray(<shape=(3, 100), dtype=float64>), ensemble_standard_error=np.ndarray(<shape=(3, 100), dtype=float64>), shape=(3, 100)), metadata={'shots': 4096, 'target_precision': 0.015625, 'circuit_metadata': {}, 'resilience': {}, 'num_randomizations': 32})], metadata={'dynamical_decoupling': {'enable': False, 'sequence_type': 'XX', 'extra_slack_distribution': 'middle', 'scheduling_method': 'alap'}, 'twirling': {'enable_gates': False, 'enable_measure': True, 'num_randomizations': 'auto', 'shots_per_randomization': 'auto', 'interleave_randomizations': True, 'strategy': 'active-accum'}, 'resilience': {'measure_mitigation': True, 'zne_mitigation': False, 'pec_mitigation': False}, 'version': 2})

The associated PubResult of this job has the following data bins:
 DataBin(evs=np.ndarray(<shape=(3, 100), dtype=float64>), stds=np.ndarray(<shape=

#### How the Estimator primitive calculates error

In addition to the estimate of the mean of the observables passed in the input PUBs (the `evs` field of the `DataBin`), Estimator also attempts to deliver an estimate of the error associated with those expectation values. All Estimator queries will populate the `stds` field with a quantity like the standard error of the mean for each expectation value, but some error mitigation options produce additional information, such as `ensemble_standard_error`.

Consider a single observable $\mathcal{O}$. In the absence of [ZNE](/docs/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne), you can think of each shot of the Estimator execution as providing a point estimate of the expectation value $\langle \mathcal{O} \rangle$. If the pointwise estimates are in a vector `Os`, then the value returned in `ensemble_standard_error` is equivalent to the following (in which $\sigma_{\mathcal{O}}$ is the [standard deviation of the expectation value](/docs/api/qiskit/qiskit.primitives.BackendEstimatorV2) estimate and $N_{shots}$ is the number of shots):

$$\frac{ \sigma_{\mathcal{O}} }{ \sqrt{N_{shots}} },$$

which treats all shots as part of a single ensemble. If you requested gate [twirling](/docs/guides/error-mitigation-and-suppression-techniques#pauli-twirling) (`twirling.enable_gates = True`), you can sort the pointwise estimates of $\langle \mathcal{O} \rangle$ into sets that share a common twirl. Call these sets of estimates `O_twirls`, and there are `num_randomizations` (number of twirls) of them. Then `stds` is the standard error of the mean of `O_twirls`, as in

$$\frac{ \sigma_{\mathcal{O}} }{ \sqrt{N_{twirls}} },$$

where $\sigma_{\mathcal{O}}$ is the standard deviation of `O_twirls` and $N_{twirls}$ is the number of twirls. When you do not enable twirling, `stds` and `ensemble_standard_error` are equal.

If you enable ZNE, then the `stds` described above become weights in a non-linear regression to an extrapolator model. What finally gets returned in the `stds` in this case is the uncertainty of the fit model evaluated at a noise factor of zero. When there is a poor fit, or large uncertainty in the fit, the reported `stds` can become very large. When ZNE is enabled, `pub_result.data.evs_noise_factors` and `pub_result.data.stds_noise_factors` are also populated, so that you can do your own extrapolation.

## Result metadata

In addition to the execution results, both the `PrimitiveResult` and `PubResult` objects contain a metadata attribute about the job that was submitted. The metadata containing information for all submitted PUBs (such as the various [runtime options](/docs/api/qiskit-ibm-runtime/options) available) can be found in the `PrimitiveResult.metatada`, while the metadata specific to each PUB is found in `PubResult.metadata`.

<Admonition type="note">
In the metadata field, primitive implementations can return any information about execution that is relevant to them, and there are no key-value pairs that are guaranteed by the base primitive. Thus, the returned metadata might be different in different primitive implementations.
</Admonition>

In [3]:
# Print out the results metadata
print("The metadata of the PrimitiveResult is:")
for key, val in result.metadata.items():
    print(f"'{key}' : {val},")

print("\nThe metadata of the PubResult result is:")
for key, val in result[0].metadata.items():
    print(f"'{key}' : {val},")

The metadata of the PrimitiveResult is:
'dynamical_decoupling' : {'enable': False, 'sequence_type': 'XX', 'extra_slack_distribution': 'middle', 'scheduling_method': 'alap'},
'twirling' : {'enable_gates': False, 'enable_measure': True, 'num_randomizations': 'auto', 'shots_per_randomization': 'auto', 'interleave_randomizations': True, 'strategy': 'active-accum'},
'resilience' : {'measure_mitigation': True, 'zne_mitigation': False, 'pec_mitigation': False},
'version' : 2,

The metadata of the PubResult result is:
'shots' : 4096,
'target_precision' : 0.015625,
'circuit_metadata' : {},
'resilience' : {},
'num_randomizations' : 32,
